In [ ]:
# Database Exploration - Interactive Notebook
# Organized by table for expert review

import pandas as pd
import psycopg2
from dotenv import load_dotenv
import os

# Load environment
#load_dotenv('.env.dev')
load_dotenv('.env')

# Create connection
conn = psycopg2.connect(
    dbname=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD', ''),
    host=os.getenv('DB_HOST'),
    port=os.getenv('DB_PORT')
)

print(f"Connected to: {os.getenv('DB_NAME')}")

In [ ]:


# ============================================================================
# DATABASE SCHEMA OVERVIEW
# ============================================================================

# List all tables
tables = pd.read_sql("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
    ORDER BY table_name
""", conn)

print("\nTables in database:")
display(tables)

# List all views
views = pd.read_sql("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public' AND table_type = 'VIEW'
    ORDER BY table_name
""", conn)

print("\nViews in database:")
display(views)

In [ ]:
# ============================================================================
# TABLE 1: IMAGES
# ============================================================================

# Schema
images_schema = pd.read_sql("""
    SELECT column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_name = 'images'
    ORDER BY ordinal_position
""", conn)

print("\n--- IMAGES TABLE ---")
print("\nSchema:")
display(images_schema)

# Row count by source
images_count = pd.read_sql("""
    SELECT source, COUNT(*) as count
    FROM images
    GROUP BY source
    ORDER BY source
""", conn)

print("\nRow count by source:")
display(images_count)

# Sample rows
images_sample = pd.read_sql("""
    SELECT *
    FROM images
    ORDER BY image_id
    LIMIT 5
""", conn)

print("\nSample rows:")
display(images_sample)

# Check for nulls
images_nulls = pd.read_sql("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(file_path) as file_path_not_null,
        COUNT(*) - COUNT(file_path) as file_path_null
    FROM images
""", conn)

print("\nNull values:")
display(images_nulls)

# Check for duplicate filenames
images_duplicates = pd.read_sql("""
    SELECT filename, COUNT(*) as count
    FROM images
    GROUP BY filename
    HAVING COUNT(*) > 1
""", conn)

print("\nDuplicate filenames:")
display(images_duplicates)


In [ ]:
# ============================================================================
# TABLE 2: GROUND_TRUTH_HISTORY
# ============================================================================

gt_schema = pd.read_sql("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_name = 'ground_truth_history'
    ORDER BY ordinal_position
""", conn)

print("\n--- GROUND_TRUTH_HISTORY TABLE ---")
print("\nSchema:")
display(gt_schema)

# Row counts
gt_counts = pd.read_sql("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE is_current = TRUE) as current_labels,
        COUNT(*) FILTER (WHERE is_current = FALSE) as historical_labels
    FROM ground_truth_history
""", conn)

print("\nRow counts:")
display(gt_counts)

# Label distribution (current only)
gt_labels = pd.read_sql("""
    SELECT label_name, value, COUNT(*) as count
    FROM ground_truth_history
    WHERE is_current = TRUE
    GROUP BY label_name, value
    ORDER BY label_name, value
""", conn)

print("\nLabel distribution (current):")
display(gt_labels)

# Sample rows
gt_sample = pd.read_sql("""
    SELECT *
    FROM ground_truth_history
    WHERE is_current = TRUE
    ORDER BY history_id
    LIMIT 10
""", conn)

print("\nSample current labels:")
display(gt_sample)

# Check uniqueness constraint (should be 1 per image/label)
gt_uniqueness = pd.read_sql("""
    SELECT image_id, label_name, COUNT(*) as count
    FROM ground_truth_history
    WHERE is_current = TRUE
    GROUP BY image_id, label_name
    HAVING COUNT(*) > 1
""", conn)

print("\nViolations of uniqueness (should be empty):")
display(gt_uniqueness)

In [ ]:
# ============================================================================
# TABLE 3: ANALYSIS_RUNS
# ============================================================================

runs_schema = pd.read_sql("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_name = 'analysis_runs'
    ORDER BY ordinal_position
""", conn)

print("\n--- ANALYSIS_RUNS TABLE ---")
print("\nSchema:")
display(runs_schema)

# All runs
runs_all = pd.read_sql("""
    SELECT *
    FROM analysis_runs
    ORDER BY run_timestamp DESC
""", conn)

print("\nAll analysis runs:")
display(runs_all)

# Count by analysis type
runs_by_type = pd.read_sql("""
    SELECT analysis_type, COUNT(*) as count
    FROM analysis_runs
    GROUP BY analysis_type
    ORDER BY analysis_type
""", conn)

print("\nRuns by analysis type:")
display(runs_by_type)

# Null check
runs_nulls = pd.read_sql("""
    SELECT 
        COUNT(*) as total,
        COUNT(python_script) as has_script,
        COUNT(model_version) as has_version,
        COUNT(hyperparameters) as has_hyperparams,
        COUNT(notes) as has_notes,
        COUNT(autoencoder_name) as has_autoencoder,
        COUNT(clustering_name) as has_clustering
    FROM analysis_runs
""", conn)

print("\nNon null value summary:")
display(runs_nulls)

In [ ]:
# ============================================================================
# TABLE 4: PREDICTIONS
# ============================================================================

pred_schema = pd.read_sql("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_name = 'predictions'
    ORDER BY ordinal_position
""", conn)

print("\n--- PREDICTIONS TABLE ---")
print("\nSchema:")
display(pred_schema)

# Row count by analysis run
pred_counts = pd.read_sql("""
    SELECT 
        ar.analysis_run_id,
        ar.run_timestamp,
        ar.analysis_type,
        ar.model_name,
        COUNT(p.prediction_id) as num_predictions
    FROM analysis_runs ar
    LEFT JOIN predictions p ON ar.analysis_run_id = p.analysis_run_id
    GROUP BY ar.analysis_run_id, ar.run_timestamp, ar.analysis_type, ar.model_name
    ORDER BY ar.run_timestamp DESC
""", conn)

print("\nPredictions per analysis run:")
display(pred_counts)

# Sample predictions with image info
pred_sample = pd.read_sql("""
    SELECT 
        p.prediction_id,
        p.analysis_run_id,
        i.source,
        i.source_image_id,
        i.filename,
        p.label_name,
        p.predicted_value,
        p.confidence_score
    FROM predictions p
    JOIN images i ON p.image_id = i.image_id
    ORDER BY p.prediction_id
    LIMIT 10
""", conn)

print("\nSample predictions:")
display(pred_sample)

# Prediction label distribution
pred_labels = pd.read_sql("""
    SELECT label_name, predicted_value, COUNT(*) as count
    FROM predictions
    GROUP BY label_name, predicted_value
    ORDER BY label_name, predicted_value
""", conn)

print("\nPrediction label distribution:")
display(pred_labels)

In [ ]:
# ============================================================================
# TABLE 5: LLM_RESPONSES
# ============================================================================

llm_schema = pd.read_sql("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_name = 'llm_responses'
    ORDER BY ordinal_position
""", conn)

print("\n--- LLM_RESPONSES TABLE ---")
print("\nSchema:")
display(llm_schema)

# Row count
llm_count = pd.read_sql("""
    SELECT COUNT(*) as total_responses
    FROM llm_responses
""", conn)

print("\nTotal LLM responses:")
display(llm_count)

# Responses by analysis run
llm_by_run = pd.read_sql("""
    SELECT 
        ar.analysis_run_id,
        ar.run_timestamp,
        ar.model_name,
        COUNT(lr.response_id) as num_responses
    FROM analysis_runs ar
    LEFT JOIN llm_responses lr ON ar.analysis_run_id = lr.analysis_run_id
    WHERE ar.analysis_type = 'llm_classification'
    GROUP BY ar.analysis_run_id, ar.run_timestamp, ar.model_name
    ORDER BY ar.run_timestamp DESC
""", conn)

print("\nLLM responses by run:")
display(llm_by_run)

# Sample responses
llm_sample = pd.read_sql("""
    SELECT 
        lr.response_id,
        lr.analysis_run_id,
        i.source,
        i.source_image_id,
        lr.parse_success
    FROM llm_responses lr
    JOIN images i ON lr.image_id = i.image_id
    ORDER BY lr.response_id
    LIMIT 5
""", conn)

print("\nSample LLM responses:")
display(llm_sample)

# Parse success rate
llm_parse_rate = pd.read_sql("""
    SELECT 
        parse_success,
        COUNT(*) as count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as percentage
    FROM llm_responses
    GROUP BY parse_success
""", conn)

print("\nParse success rate:")
display(llm_parse_rate)

In [ ]:

# ============================================================================
# TABLE 6: CLUSTERING_RESULTS
# ============================================================================

cluster_schema = pd.read_sql("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_name = 'clustering_results'
    ORDER BY ordinal_position
""", conn)

print("\n--- CLUSTERING_RESULTS TABLE ---")
print("\nSchema:")
display(cluster_schema)

# Row count
cluster_count = pd.read_sql("""
    SELECT COUNT(*) as total_clusters
    FROM clustering_results
""", conn)

print("\nTotal clustering results:")
display(cluster_count)

# Cluster distribution
cluster_dist = pd.read_sql("""
    SELECT cluster_id, COUNT(*) as count
    FROM clustering_results
    GROUP BY cluster_id
    ORDER BY cluster_id
""", conn)

print("\nCluster distribution:")
display(cluster_dist)

# Sample with image info
cluster_sample = pd.read_sql("""
    SELECT 
        cr.clustering_id,
        cr.analysis_run_id,
        i.source,
        i.source_image_id,
        cr.cluster_id
    FROM clustering_results cr
    JOIN images i ON cr.image_id = i.image_id
    ORDER BY cr.cluster_id
    LIMIT 10
""", conn)

print("\nSample clustering results:")
display(cluster_sample)

In [ ]:
# ============================================================================
# TABLE 7: PROMPTS
# ============================================================================

prompts_schema = pd.read_sql("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_name = 'prompts'
    ORDER BY ordinal_position
""", conn)

print("\n--- PROMPTS TABLE ---")
print("\nSchema:")
display(prompts_schema)

# All prompts
prompts_all = pd.read_sql("""
    SELECT *
    FROM prompts
    ORDER BY prompt_id
""", conn)

print("\nAll prompts:")
display(prompts_all)


In [ ]:
# ============================================================================
# CROSS-TABLE QUERIES & DATA QUALITY
# ============================================================================

print("\n--- DATA QUALITY CHECKS ---")

# Images without ground truth
images_no_gt = pd.read_sql("""
    SELECT i.source, COUNT(DISTINCT i.image_id) as images_without_ground_truth
    FROM images i
    LEFT JOIN ground_truth_history gt ON i.image_id = gt.image_id AND gt.is_current = TRUE
    WHERE gt.image_id IS NULL
    GROUP BY i.source
""", conn)

print("\nImages without ground truth:")
display(images_no_gt)

# Images without predictions
images_no_pred = pd.read_sql("""
    SELECT i.source, COUNT(DISTINCT i.image_id) as images_without_predictions
    FROM images i
    LEFT JOIN predictions p ON i.image_id = p.image_id
    WHERE p.image_id IS NULL
    GROUP BY i.source
""", conn)

print("\nImages without predictions:")
display(images_no_pred)

# Predictions without ground truth
pred_no_gt = pd.read_sql("""
    SELECT COUNT(*) as predictions_without_ground_truth
    FROM predictions p
    LEFT JOIN ground_truth_history gt 
        ON p.image_id = gt.image_id 
        AND p.label_name = gt.label_name 
        AND gt.is_current = TRUE
    WHERE gt.image_id IS NULL
""", conn)

print("\nPredictions without ground truth:")
display(pred_no_gt)

In [ ]:
# ============================================================================
# EXAMPLE JOINS
# ============================================================================

print("\n--- EXAMPLE QUERIES ---")

# Compare predictions to ground truth for one run
comparison = pd.read_sql("""
    SELECT 
        i.source,
        i.source_image_id,
        p.label_name,
        gt.value as ground_truth,
        p.predicted_value,
        p.confidence_score,
        CASE 
            WHEN gt.value = p.predicted_value THEN 'correct'
            ELSE 'incorrect'
        END as match
    FROM predictions p
    JOIN images i ON p.image_id = i.image_id
    JOIN ground_truth_history gt 
        ON p.image_id = gt.image_id 
        AND p.label_name = gt.label_name 
        AND gt.is_current = TRUE
    WHERE p.analysis_run_id = (
        SELECT MAX(analysis_run_id) FROM analysis_runs WHERE analysis_type = 'yolo_classification'
    )
    LIMIT 10
""", conn)

print("\nSample prediction vs ground truth comparison (latest YOLO run):")
display(comparison)

# Full image summary with all data types
image_summary = pd.read_sql("""
    SELECT 
        i.image_id,
        i.source,
        i.source_image_id,
        i.filename,
        COUNT(DISTINCT CASE WHEN gt.is_current THEN gt.label_name END) as num_labels,
        COUNT(DISTINCT p.analysis_run_id) as num_prediction_runs,
        COUNT(DISTINCT lr.response_id) as num_llm_responses,
        COUNT(DISTINCT cr.cluster_id) as num_cluster_assignments
    FROM images i
    LEFT JOIN ground_truth_history gt ON i.image_id = gt.image_id
    LEFT JOIN predictions p ON i.image_id = p.image_id
    LEFT JOIN llm_responses lr ON i.image_id = lr.image_id
    LEFT JOIN clustering_results cr ON i.image_id = cr.image_id
    GROUP BY i.image_id, i.source, i.source_image_id, i.filename
    ORDER BY i.image_id
    LIMIT 10
""", conn)

print("\nImage summary (all data types):")
display(image_summary)

# Close connection
conn.close()
print("\nConnection closed")

In [ ]:
# Load environment
#load_dotenv('.env.dev')
load_dotenv('.env')

# Create connection
conn = psycopg2.connect(
    dbname=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD', ''),
    host=os.getenv('DB_HOST'),
    port=os.getenv('DB_PORT')
)

print(f"Connected to: {os.getenv('DB_NAME')}")


In [ ]:



comparison = pd.read_sql("""
    SELECT 
        i.source,
        i.source_image_id,
        p.label_name,
        p.predicted_value
    FROM predictions p
    JOIN images i ON p.image_id = i.image_id
    WHERE p.predicted_value is NULL
    LIMIT 10
""", conn)

print("\nSample prediction vs ground truth comparison (latest YOLO run):")
display(comparison)

In [ ]:
output = pd.read_sql("""
    SELECT 
        distinct label_name
    FROM predictions
""", conn)

print("\nResults")
display(output)

In [ ]:
output = pd.read_sql("""
    SELECT 
        *
    FROM ground_truth_wide
""", conn)

print("\nResults")
display(output)

In [ ]:
# Responses by analysis run
llm_by_run = pd.read_sql("""
    SELECT 
        ar.analysis_run_id,
        ar.run_timestamp,
        ar.model_name,
        COUNT(lr.response_id) as num_responses
    FROM analysis_runs ar
    LEFT JOIN llm_responses lr ON ar.analysis_run_id = lr.analysis_run_id
    WHERE ar.analysis_type = 'llm_classification'
    GROUP BY ar.analysis_run_id, ar.run_timestamp, ar.model_name
    ORDER BY ar.run_timestamp DESC
""", conn)

print("\nLLM responses by run:")
display(llm_by_run)

In [ ]:
# Row count
llm_count = pd.read_sql("""
    SELECT COUNT(*) as total_responses
    FROM llm_responses
""", conn)

print("\nTotal LLM responses:")
display(llm_count)

In [ ]:
# Close connection
conn.close()
print("\nConnection closed")